In [192]:
# Import dependencies 
import pandas as pd
import numpy as np
import ast
import pickle
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [190]:
!pip install pickle

ERROR: Could not find a version that satisfies the requirement pickle (from versions: none)
ERROR: No matching distribution found for pickle


### Strategy

Create a new data frame with only two fields once should be the title(Original_title) and other one should be tags

combination of columns to create tags
title_y, genres, story, summery, actors, taglines, 

sorting results
imdb_rating


In [214]:
df = pd.read_csv('movies.csv')
df.head(1)

,title_x,imdb_id,poster_path,wiki_link,title_y,original_title,is_adult,year_of_release,runtime,genres,imdb_rating,imdb_votes,story,summary,tagline,actors,wins_nominations,release_date
0,Uri: The Surgical Strike,tt8291224,https://upload.wikimedia.org/wikipedia/en/thum...,https://en.wikipedia.org/wiki/Uri:_The_Surgica...,Uri: The Surgical Strike,Uri: The Surgical Strike,0,2019,138,Action|Drama|War,8.4,35112,Divided over five chapters the film chronicle...,Indian army special forces execute a covert op...,NaN,Vicky Kaushal|Paresh Rawal|Mohit Raina|Yami Ga...,4 wins,11 January 2019 (USA)


In [215]:
# Filling missing values with empyt string ("")
df['story'] = df['story'].fillna('')
df['actors'] = df['actors'].fillna('')
df['tagline'] = df['tagline'].fillna('')
df['poster_path'] = df['poster_path'].fillna('')

In [216]:
df['actors'] = df['actors'].apply(lambda x: " ".join(x.split('|')[:3]))
df['genres'] = df['genres'].apply(lambda x: x.replace('|', " "))

In [ ]:
df.isnull().sum()

In [ ]:
# Creating functions to compare the values in two different fields.
def compare_data(col1,col2):
    dict1 = {}
    for i,v in enumerate(col1):
        if v.lower()==col2[i].lower():
            pass
        else:
            dict1[v] = col2[i]
    return dict1

# Calling 
compare_data(df['original_title'], df['title_y'])

In [218]:
df = df[['original_title', 'title_y', 'genres', 'story', 'summary', 'actors', 'tagline', 'imdb_rating', 'poster_path']]
df['tags'] = df['genres'] + " " + df['story'] + " " + df['summary'] + " " + df['actors'] + " " + df['tagline'] + " " + df['title_y']
df.isnull().sum()

original_title    0
title_y           0
genres            0
story             0
summary           0
actors            0
tagline           0
imdb_rating       0
poster_path       0
tags              0
dtype: int64

In [219]:
movies_df = pd.DataFrame({
    'title': df['original_title'],
    'poster_path': df['poster_path'],
    'tags': df['tags']
})

movies_df['tags'] = movies_df['tags'].str.lower()

In [207]:
movies_df.isnull().sum()

title            0
poster_path    103
tags             0
dtype: int64

### Converting the tags into words

In [209]:
vectors = TfidfVectorizer(max_features=5000, stop_words='english')

In [210]:
vec = vectors.fit_transform(movies_df['tags']).toarray()

In [ ]:
print(list(vectors.get_feature_names_out()))

### Steaming the words

In [220]:
ps = PorterStemmer()

def stem(text):
    steming = []
    for item in text.split():
        steming.append(ps.stem(item))
    return " ".join(steming)

movies_df['tags'] = movies_df['tags'].apply(stem)

In [211]:
similarity = cosine_similarity(vec)

In [212]:
def recommend(movie):
    movie_index = movies_df[movies_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_lst = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    print(movies_lst)
    for i in movies_lst:
        print(movies_df.iloc[i[0]].title)
    
recommend('Uri: The Surgical Strike')

[(219, np.float64(0.23792360543966548)), (1, np.float64(0.22265228429264444)), (1488, np.float64(0.21537161281571662)), (1314, np.float64(0.2105867503096873)), (1536, np.float64(0.20998623271852107))]
Raag Desh
Battalion 609
Zameen
Main Hoon Na
Maa Tujhhe Salaam


In [169]:
# movies_df[movies_df['title'] == 'Why Cheat India'].index[0]
sorted(list(enumerate(similarity[0])), reverse=True, key=lambda x: x[1])[1:6]

[(219, np.float64(0.23792360543966548)),
 (1, np.float64(0.22265228429264444)),
 (1488, np.float64(0.21537161281571662)),
 (1314, np.float64(0.2105867503096873)),
 (1536, np.float64(0.20998623271852107))]

In [221]:
pickle.dump(movies_df,open('movies.pkl','wb'))
pickle.dump(similarity,open('similarity.pkl','wb'))

0            Uri: The Surgical Strike
1                       Battalion 609
2       The Accidental Prime Minister
3                     Why Cheat India
4                     Evening Shadows
                    ...              
1624            Tera Mera Saath Rahen
1625             Yeh Zindagi Ka Safar
1626                  Sabse Bada Sukh
1627                            Daaka
1628                         Humsafar
Name: title, Length: 1629, dtype: object